In [1]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import csv

# --------- Configuration ---------
DATA_DIR     = r"E:\Learning\UNSW\Term2\9444\group_project\data\split_with_713"
OUTPUT_DIR   = r"E:\Learning\UNSW\Term2\9444\group_project\outputs\plot\cnn9_base_with_trans"
NUM_CLASSES  = 39
BATCH_SIZE   = 64
NUM_WORKERS  = 4
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS   = 30
WEIGHT_DECAY = 1e-4
LR           = 1e-3
PATIENCE     = 5

#LIMIT_TRAIN = None
#LIMIT_VAL   = None
#LIMIT_TEST  = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.backends.cudnn.benchmark = True

# --------- Transforms ---------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# --------- Dataloaders ---------
def get_dataloaders(data_dir, batch_size, num_workers, limit_train=None, limit_val=None, limit_test=None):
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
    val_ds   = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transform)
    test_ds  = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

    if limit_train: train_ds = Subset(train_ds, range(min(limit_train, len(train_ds))))
    if limit_val:   val_ds = Subset(val_ds, range(min(limit_val, len(val_ds))))
    if limit_test:  test_ds = Subset(test_ds, range(min(limit_test, len(test_ds))))

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True),
    )

# --------- CNN + Transformer Model ---------
class CNNTransformer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # CNN Feature Extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.embedding_dim = 128

        # Flatten spatial grid to patch tokens (56x56 -> 3136 patches)
        self.seq_len = 56 * 56
        self.linear_proj = nn.Linear(self.embedding_dim, 128)

        # Transformer Encoder (1 layer, 4 heads)
        encoder_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=256, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)  # [B, 128, 56, 56]
        B, C, H, W = x.shape
        x = x.view(B, C, -1).permute(0, 2, 1)  # [B, HW, C]
        x = self.linear_proj(x)               # [B, HW, 128]
        x = self.transformer(x)               # [B, HW, 128]
        x = x.mean(dim=1)                     # global average pooling over tokens
        return self.classifier(x)

model = CNNTransformer(NUM_CLASSES).to(DEVICE)

# --------- Optimizer & Loss ---------
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda')

# --------- Evaluation ---------
def evaluate(model, loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().tolist()
            preds_all.extend(preds)
            labels_all.extend(labels.tolist())
    return accuracy_score(labels_all, preds_all)

# --------- Training Loop ---------
def train():
    #train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, BATCH_SIZE, NUM_WORKERS, LIMIT_TRAIN, LIMIT_VAL, LIMIT_TEST)
    train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, BATCH_SIZE, NUM_WORKERS)
    best_val_acc = 0.0
    epochs_no_improve = 0
    
    train_losses = []
    val_accuracies = []
    epochs_list = []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)

        scheduler.step()
        val_acc = evaluate(model, val_loader)
        train_loss = running_loss / len(train_loader.dataset)
        
        train_losses.append(train_loss)
        val_accuracies.append(val_acc)
        epochs_list.append(epoch)
        
        print(f"Epoch {epoch}/{NUM_EPOCHS} - Train Loss: {train_loss:.4f} - Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_cnn_with_trans.pth"))
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    test_acc = evaluate(model, test_loader)
    print(f"Best Val Acc: {best_val_acc:.4f} - Test Acc: {test_acc:.4f}")
    
    # --------- Save metrics CSV ---------
    metrics_path = os.path.join(OUTPUT_DIR, 'metrics.csv')
    with open(metrics_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_acc'])
        for e, l, a in zip(epochs_list, train_losses, val_accuracies):
            writer.writerow([e, l, a])

    # --------- Plot and save figures ---------
    # Training Loss
    fig1 = plt.figure()
    plt.plot(epochs_list, train_losses, marker='o')
    plt.title('Training Loss vs. Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    fig1.savefig(os.path.join(OUTPUT_DIR, 'training_loss.png'), bbox_inches='tight')
    plt.close(fig1)

    # Validation Accuracy
    fig2 = plt.figure()
    plt.plot(epochs_list, val_accuracies, marker='o')
    plt.title('Validation Accuracy vs. Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    fig2.savefig(os.path.join(OUTPUT_DIR, 'validation_accuracy.png'), bbox_inches='tight')
    plt.close(fig2)

if __name__ == "__main__":
    train()


Epoch 1/30 - Train Loss: 2.8658 - Val Acc: 0.3352
Epoch 2/30 - Train Loss: 2.0728 - Val Acc: 0.5486
Epoch 3/30 - Train Loss: 1.4644 - Val Acc: 0.7153
Epoch 4/30 - Train Loss: 1.1330 - Val Acc: 0.7981
Epoch 5/30 - Train Loss: 0.9465 - Val Acc: 0.8193
Epoch 6/30 - Train Loss: 0.8199 - Val Acc: 0.8816
Epoch 7/30 - Train Loss: 0.7315 - Val Acc: 0.8764
Epoch 8/30 - Train Loss: 0.6599 - Val Acc: 0.8818
Epoch 9/30 - Train Loss: 0.6094 - Val Acc: 0.9204
Epoch 10/30 - Train Loss: 0.5590 - Val Acc: 0.8987
Epoch 11/30 - Train Loss: 0.5080 - Val Acc: 0.9102
Epoch 12/30 - Train Loss: 0.4765 - Val Acc: 0.9348
Epoch 13/30 - Train Loss: 0.4514 - Val Acc: 0.9293
Epoch 14/30 - Train Loss: 0.4259 - Val Acc: 0.9437
Epoch 15/30 - Train Loss: 0.3860 - Val Acc: 0.9250
Epoch 16/30 - Train Loss: 0.3638 - Val Acc: 0.9551
Epoch 17/30 - Train Loss: 0.3367 - Val Acc: 0.9611
Epoch 18/30 - Train Loss: 0.3212 - Val Acc: 0.9601
Epoch 19/30 - Train Loss: 0.2998 - Val Acc: 0.9635
Epoch 20/30 - Train Loss: 0.2829 - Val A